# EmporiUm Sales Territory Analysis

## Capstone 2: Business Analysis with Python

**Assigned territories:** Massachusetts and New Jersey  
**Region:** Northeast

This notebook analyzes in-store sales performance for the assigned territories using pandas, NumPy, and Matplotlib.

## Step 1: Import Libraries

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## Step 2: Load CSV Files

All CSV files should be in the same folder as this notebook.

In [ ]:

store_sales = pd.read_csv('StoreSales.csv')
store_sales.info()


In [ ]:
store_sales.head()

In [ ]:

store_detail = pd.read_csv('StoreDetail.csv')
store_detail.info()


In [ ]:
store_detail.head()

In [ ]:

products = pd.read_csv('Products.csv')
products.info()


In [ ]:
products.head()

In [ ]:

product_categories = pd.read_csv('ProductCategories.csv')
product_categories.info()


In [ ]:
product_categories.head()

In [ ]:

# customer_list.csv is pipe-separated, so sep='|' is required.
customers = pd.read_csv('customer_list.csv', sep='|')
customers.columns = customers.columns.str.strip()
customers.info()


In [ ]:
customers.head()

## Step 3: Clean and Merge Data

In [ ]:

# Clean dates and IDs
store_sales['Transaction Date'] = pd.to_datetime(store_sales['Transaction Date'])
store_sales['Month'] = store_sales['Transaction Date'].dt.to_period('M').astype(str)
store_sales['RewardsID'] = pd.to_numeric(store_sales['RewardsID'], errors='coerce')
customers['cust_id'] = pd.to_numeric(customers['cust_id'], errors='coerce')

# Merge sales with store details
sales_store = pd.merge(
    store_sales,
    store_detail,
    on='Store ID',
    how='left'
)

# Filter to assigned territories/states
assigned_states = ['Massachusetts', 'New Jersey']
territory_sales = sales_store[sales_store['State'].isin(assigned_states)].copy()

territory_sales.head()


In [ ]:

# Merge product and category information for territory sales
sales_products = pd.merge(
    territory_sales,
    products,
    on='Prod Num',
    how='left'
)

full_sales = pd.merge(
    sales_products,
    product_categories,
    on=['CategoryID', 'SubcategoryID'],
    how='left'
)

full_sales.head()


# Core Marketing Analysis

## 1. Territory Managers, Store IDs, and Cities

In [ ]:

territory_info = store_detail[store_detail['State'].isin(assigned_states)][
    ['State', 'Territory Manager', 'Region', 'Region Director', 'Store ID', 'Store Location']
].sort_values(['State', 'Store ID'])

territory_info


## 2. Monthly Total Revenue by Territory

In [ ]:

monthly_revenue = territory_sales.groupby(['State', 'Month'], as_index=False)['Sale Amount'].sum()
monthly_revenue = monthly_revenue.sort_values(['State', 'Month'])
monthly_revenue


In [ ]:

plt.figure(figsize=(12, 6))

for state in assigned_states:
    state_data = monthly_revenue[monthly_revenue['State'] == state]
    plt.plot(state_data['Month'], state_data['Sale Amount'], marker='o', label=state)

plt.title('Monthly Revenue: Massachusetts vs New Jersey')
plt.xlabel('Month')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


## 3. Store Performance Rankings

In [ ]:

store_performance = territory_sales.groupby(
    ['State', 'Store ID', 'Store Location'], as_index=False
).agg(
    Total_Revenue=('Sale Amount', 'sum'),
    Transactions=('Sale Amount', 'count'),
    Average_Sale=('Sale Amount', 'mean')
).sort_values(['State', 'Total_Revenue'], ascending=[True, False])

store_performance


In [ ]:

plt.figure(figsize=(12, 6))

store_chart = store_performance.sort_values('Total_Revenue', ascending=False).head(15)
plt.bar(store_chart['Store Location'] + ', ' + store_chart['State'], store_chart['Total_Revenue'])

plt.title('Top 15 Stores by Revenue')
plt.xlabel('Store')
plt.ylabel('Total Revenue')
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()


## 4. Top Rewards Customers in Each Territory

In [ ]:

customer_sales = pd.merge(
    territory_sales,
    customers,
    left_on='RewardsID',
    right_on='cust_id',
    how='left'
)

top_customers = customer_sales.dropna(subset=['cust_id']).groupby(
    ['State', 'cust_id', 'name', 'email'], as_index=False
).agg(
    Total_Spent=('Sale Amount', 'sum'),
    Transactions=('Sale Amount', 'count')
).sort_values(['State', 'Total_Spent'], ascending=[True, False])

top_customers.groupby('State').head(10)


## 5. Product Category Transactions and Revenue by Month

In [ ]:

category_monthly = full_sales.groupby(
    ['State', 'Month', 'Category'], as_index=False
).agg(
    Transactions=('Sale Amount', 'count'),
    Total_Revenue=('Sale Amount', 'sum')
).sort_values(['State', 'Month', 'Category'])

category_monthly


In [ ]:

category_totals = full_sales.groupby(['State', 'Category'], as_index=False).agg(
    Transactions=('Sale Amount', 'count'),
    Total_Revenue=('Sale Amount', 'sum')
).sort_values(['State', 'Total_Revenue'], ascending=[True, False])

category_totals


In [ ]:

plt.figure(figsize=(10, 6))

category_by_state = category_totals.pivot(index='Category', columns='State', values='Total_Revenue').fillna(0)
category_by_state.plot(kind='bar', figsize=(12, 6))

plt.title('Revenue by Product Category and State')
plt.xlabel('Product Category')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Marketing Recommendation

Based on the analysis, marketing attention should focus on the strongest revenue categories and the stores with room for growth in Massachusetts and New Jersey. Massachusetts is managed by **Bo Heap**, while New Jersey is managed by **Miami Vue**. Both states are part of the **Northeast** region under **Michael Jarvis**.

Recommended next steps:

1. Use high-performing product categories as the center of the next campaign.
2. Promote loyalty/rewards campaigns to increase repeat customer spending.
3. Compare lower-performing stores against top stores to identify where local marketing support is needed.
4. Use monthly revenue trends to time promotions around stronger buying periods.

The clearest opportunity is to combine product-category marketing with customer rewards targeting so that EmporiUm can increase both transaction count and average customer spend.

# Bonus Analysis Starter

In [ ]:

# January 2024 transactions over $500 in assigned territories
january_2024 = territory_sales[
    (territory_sales['Transaction Date'].dt.year == 2024) &
    (territory_sales['Transaction Date'].dt.month == 1)
].sort_values('Sale Amount', ascending=False)

january_2024_over_500 = january_2024[january_2024['Sale Amount'] > 500]
january_2024_over_500.head()


In [ ]:

# Top 10 products by total revenue in assigned territories
top_products = full_sales.groupby(['Prod Num', 'Product'], as_index=False).agg(
    Total_Revenue=('Sale Amount', 'sum'),
    Transactions=('Sale Amount', 'count')
).sort_values('Total_Revenue', ascending=False)

top_products.head(10)
